# Session 3 — Exercises

**Before session 4.** About ninety minutes.

Same house pattern: **predict, then measure.** Write down what you expect before you run anything. In
this session you have something session 2 did not — a *number* — so "I think this will help" can be
checked rather than believed.

Answers go in the **✍️ Your answer** cells. Code TODOs are marked `# TODO`.

Solutions are released with session 4, after these have been reviewed in class.

## Setup — given, just run it

In [ ]:
import json
import re
import subprocess
import sys
from pathlib import Path
from typing import Literal, Optional

import yaml
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))
from config import settings                      # noqa: E402
from schemas import TicketCategory, TriageResult  # noqa: E402

client = genai.Client()
TICKETS = yaml.safe_load((settings.DATA_DIR / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in TICKETS}

TRIAGE_PROMPT = yaml.safe_load(
    (HERE / "project" / "prompts.yml").read_text(encoding="utf-8"))["triage_prompt"]


def block(ticket: dict) -> str:
    return f"<ticket>\nSubject: {ticket['subject']}\n{ticket['body']}\n</ticket>"


def ask(schema, ticket):
    """The session's call, in one place. Returns a validated object — or None."""
    r = client.models.generate_content(
        model=settings.MODEL,
        contents=block(ticket),
        config=types.GenerateContentConfig(
            system_instruction=TRIAGE_PROMPT,
            response_mime_type="application/json",
            response_schema=schema))
    return r.parsed


print(f"{len(TICKETS)} tickets · model {settings.MODEL} · key {settings.has_key}")

## ⭐ Exercise 1 — The confidence field

**What this is really teaching:** that a model can tell you when it is unsure *if you give it
somewhere to say so* — and that whether the signal is any good is an empirical question, not a
design one. Section 7.2 of the notebook set this up.

Add `confidence: Literal["low", "medium", "high"]` to the schema with a `Field(description=...)` that
says **when** to answer low. Run it on `TCK-003` — the ticket section 8.1 argues about, where
`complaint` and `order_status` both genuinely fit — and on `TCK-001`, which is unambiguous.

**✍️ Predict first:** will it report low confidence on the ticket a *human* would hesitate over?

**✍️ Your prediction:**

*(before you run anything)*

In [ ]:
print("TCK-003:", by_id["TCK-003"]["body"][:160])
print("\nTCK-001:", by_id["TCK-001"]["body"][:160])

# TODO: define TriageWithConfidence — TriageResult plus a `confidence` field.
#       The description matters: say WHEN it should answer "low".

# TODO: run it on TCK-003 and TCK-001 and print category + confidence for both.

**✍️ Your answer.** Did confidence drop on the ambiguous one? **Be ready for it not to** — and
if it did not, that is the more useful finding: write down what it means that the model is sure about
a ticket two humans would argue over.

Then the design question that matters more: **what would your code do with `confidence == "low"`?**
Name a concrete action. If you cannot name one, the field is decoration you pay output tokens for on
every call — say so, and say what you would do instead.

*(your answer here)*

## ⭐⭐ Exercise 2 — The angry detector

**What this is really teaching:** branching in **code** on model output. This is the whole point of
the session, in four lines.

Write `draft_reply(ticket, triage)` that prepends an apology paragraph when `sentiment == "angry"` —
in Python, with an `if`, not by asking the prompt to be sorry when appropriate.

**✍️ Predict first:** why is the `if` better than putting "apologise if they seem angry" in the
prompt? And which of the two can you write a test for?

**✍️ Your prediction:**

In [ ]:
SYSTEM = """You are the customer support assistant for ShopWise, an online electronics store.
Reply warmly and plainly. Never promise a refund, delivery date or any outcome —
say a human colleague will follow up."""

# TODO: write draft_reply(ticket, triage) -> str.
#       If triage.sentiment == "angry", prepend an apology paragraph IN CODE.

# TODO: run it on TCK-003 (angry) and TCK-001 (calm) and compare.

**✍️ Your answer.** You now have two ways to get an apology: an `if` in your code, or a
sentence in the prompt. Give one reason to prefer each. Then: which one can you **test**, and why does
that settle it for anything a customer sees?

*(your answer here)*

## ⭐⭐⭐ Exercise 3 — Build an objective metric

**What this is really teaching:** section 8.1 claimed every conventional metric needs an objective one
beside it. Here you build the objective one, and it changes what you can say about the system.

Category accuracy is a *conventional* metric: two of the twenty labels are arguable, so part of the
gap is a disagreement rather than a mistake. **Order-ID extraction is not.** The ground truth is
`re.findall(r"ORD-\d+", ticket["body"])` — a fact, checkable by a regular expression, with nothing to
argue about.

Write `score_extraction.py` beside `score_triage.py`. It should:

1. add `order_ids: list[str]` to the schema, with a description naming the `ORD-XXXX` format, saying
   what to do when there are none, and forbidding invention;
2. run over all twenty tickets;
3. report **exact-match accuracy**, and separately count two failure kinds that are not the same
   thing at all — **missed** IDs (in the text, not extracted) and **hallucinated** IDs (extracted,
   not in the text);
4. print the tickets it got wrong.

**✍️ Predict first:** which will be higher — category accuracy or extraction accuracy? And which of
missed vs hallucinated would you rather have in a system that looks orders up automatically?

**✍️ Your prediction:**

In [ ]:
# The ground truth is free — no labelling needed, which is the point of an objective metric.
for t in TICKETS:
    found = re.findall(r"ORD-\d+", t["body"])
    if len(found) != 1:
        print(f"  {t['id']}: {len(found)} order id(s) {found}")

# TODO: define TriageWithOrders — TriageResult plus order_ids: list[str].

# TODO: write score_extraction.py (a real file, beside score_triage.py) and run it here
#       with subprocess, the same way the notebook runs the scorer.

**✍️ Your answer.** Report the number. Then:

- Did it ever **invent** an ID that was not in the text? What would that cost you in session 5, when
  a tool looks each one up?
- Empty list or `None` for "no orders mentioned" — pick one and justify it from the point of view of
  the code that *consumes* the field. Write the `for` loop in your head first.
- Your extraction score and your category score disagree about how good this system is. **Which one
  would you put in front of a manager, and what would you say about the other?**

*(your answer here)*

## ⭐⭐ Exercise 4 — Earn your few-shot

**What this is really teaching:** that examples are not free, and "it feels better" is not a result.

`score_triage.py` prints which tickets are misclassified. Your job is to change the number — or to
prove you could not.

In [ ]:
done = subprocess.run([sys.executable, "score_triage.py"],
                      cwd=HERE, capture_output=True, text=True, timeout=900)
print(done.stdout)

**Part A.** Look at the misses. For each, decide: is the model wrong, or is the *label*
arguable? One line each. Use section 8.1's table to justify which kind it is.

**✍️ Your answer:**

*(one line per missed ticket)*

**Part B.** Open `project/prompts.yml` and edit the `few_shot_examples` block. Write **three**
examples aimed at the misses you just diagnosed.

> ⚠️ **Do not copy tickets out of `tickets.yml`.** They are your test set. An example that is also a
> test case will make your score go up while your assistant gets worse — section 8.3 shows exactly
> that happening. Write fresh ones.

Then re-score with the examples switched on.

In [ ]:
# TODO: run score_triage.py with --diff and read the comparison.
#       (subprocess, same shape as the cell above, plus "--diff")

**Part C.** Now remove your examples **one at a time**, re-scoring after each. Then run the
whole comparison **twice** — appendix G showed this set moving by up to three tickets between
identical runs.

**✍️ Your answer.** Which example was pulling its weight, and which was decoration? Did any of them
**break** a ticket that was previously right? Is your best result bigger than the run-to-run noise you
just measured? Finally, the engineering call: would you ship your three examples, given they are
resent on every call forever? Defend it with the number, not with a feeling.

*(your answer here)*

## ⭐⭐ Exercise 5 — The constraint that usually works

**What this is really teaching:** section 6.3's discovery is not a curiosity, it is a class of outage
— and the reason it reaches production is that it *passes your tests*. Here you reproduce that, which
is a different and more useful skill than reading about it.

The schema below asks for a summary of at most 60 characters. Gemini does not enforce that. But it
does **read** it, and mostly complies — so a naive test says everything is fine.

1. Run `TightSummary` over all twenty tickets and count `.parsed is None`. Record the rate.
2. Now vary the cap — 40, 50, 60 — and print the summary *lengths* you get back. Do they track the
   cap? What does that tell you about whether the model can see the constraint?
3. Now **break it**: add one sentence of pressure to the input (`"Be thorough and detailed in the
   summary."`) and measure the rate again.
4. Then price three fixes and measure each: (a) state the limit in the `description` as well;
   (b) relax the constraint to something the model reaches anyway; (c) the repair loop from section
   9.3.

**✍️ Predict first:** what invalid rate do you expect at step 1, and at step 3? Which of the three
fixes will you ship?

**✍️ Your prediction:**

In [ ]:
class TightSummary(BaseModel):
    category: TicketCategory
    summary: str = Field(max_length=60)      # Gemini reads this. It does not enforce it.


# TODO 1: run TightSummary over all 20 tickets. Count None. Print the invalid rate.
#         You cannot use ask() blindly here — checking for None IS the exercise.

# TODO 2: rebuild the schema at max_length 40, 50 and 60 and print the summary LENGTHS.
#         (json.loads(r.text)["summary"] gets you the string even when .parsed is None.)

# TODO 3: append "Be thorough and detailed in the summary." to the ticket block and re-measure.

# TODO 4: three fixes, three rates, three costs, side by side.

**✍️ Your answer.** Give the rates from steps 1 and 3, the length table from step 2, and
the cost of each fix. Then the judgement:

- **Why is a constraint that works 100% of the time in testing worse than one that never works?**
  Answer this one properly — it is the whole exercise.
- Which fix would you ship, and what does the choice depend on?
- One of the three makes the failure **impossible** rather than **rarer**. Which, and why is that
  category of difference worth caring about?
- Where should the None-check live so that no future call site can forget it? (`project/triage.py`
  has already made this decision — find it, and say whether you agree.)

*(your answer here)*

## 💼 Interview questions

Out loud, 45 seconds each. These are asked, in these words, in real interviews.

**1. When do you demand structured output, and when is prose the product?**

*(your answer)*

**2. What is the difference between asking a model for JSON and passing it a schema?**

*(your answer)*

**3. The model returns a category that is not in your list. Name the two places you could catch it,
and which you prefer.**

*(your answer)*

**4. Your schema says `max_length=60` and you are getting no results at all, with no errors. What
happened?**

*(your answer)*

**5. You add three few-shot examples and accuracy is unchanged. Name two things that could be wrong.**

*(your answer)*

**6. Where would you put a rule that says "a refund ticket always goes to a human" — the prompt, the
schema, or the code? Why?**

*(your answer)*

## The one from the answer key

Both of v0.2's LLM calls could have been given a schema. Only one was.

**✍️ Which call should stay free-text, and why?**

*(your answer here)*